In [74]:
import duckdb

def query_my_duckdb(query: str, show: bool = True):
    with duckdb.connect('/home/prange/Documentos/repos/dbt-lab/db/my-duckdb.duckdb') as conn:
        result = conn.sql(query)
        if show:
            result.show()
        # return result.fetchdf()  # retorna um DataFrame do pandas

In [67]:
query_str="""
    show all tables
"""
query_my_duckdb(query_str)

┌───────────┬─────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database  │ schema  │         name         │                                                                         column_names                                                                         │                                        column_types                                        │ temporary │
│  varchar  │ varchar │       varchar        │                                                                          varchar[]                                                                           │                                         varchar[]                                          │  boolean  │
├───────────┼─────────┼──────────────────────┼─────────────────────

In [68]:

query_str="""
    select * from information_schema.tables;
"""
query_my_duckdb(query_str)

┌───────────────┬──────────────┬──────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │      table_name      │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │       varchar        │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────┼────────────┼──────────────────────────────┼──────────────────────┼───────────────────────────┼──────────────────────────┼──────────────────────

In [14]:

query_str="""
    select * from raw.rh limit 10
"""
query_my_duckdb(query_str)

┌──────────────────────┬────────────────┬────────────────────────────────────────┬─────────────────┬───────────────┬─────────────────────┬─────────────────────────┬──────────────────────┬───────────────────┬──────────────────────────────────┐
│         Nome         │      CPF       │                Unidade                 │ Data_Nascimento │ Data_Admissao │ Status_Afastamento  │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │ Data_Desligamento │             filename             │
│       varchar        │    varchar     │                varchar                 │     varchar     │    varchar    │       varchar       │         varchar         │       varchar        │      varchar      │             varchar              │
├──────────────────────┼────────────────┼────────────────────────────────────────┼─────────────────┼───────────────┼─────────────────────┼─────────────────────────┼──────────────────────┼───────────────────┼──────────────────────────────────┤
│ Ana Paula Santos     │ 123

In [41]:

query_str="""
    select * from trusted.rh_trusted
"""
query_my_duckdb(query_str)

┌────────────┬──────────────────────────┬────────────────┬────────────────────────────────────────┬─────────────────┬───────────────┬───────────┬────────────────────┬─────────────────────────┬──────────────────────┬───────────────────┐
│  ref_data  │           Nome           │      CPF       │                Unidade                 │ Data_Nascimento │ Data_Admissao │  Status   │ Status_Afastamento │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │ Data_Desligamento │
│    date    │         varchar          │    varchar     │                varchar                 │      date       │     date      │  varchar  │      varchar       │          date           │         date         │       date        │
├────────────┼──────────────────────────┼────────────────┼────────────────────────────────────────┼─────────────────┼───────────────┼───────────┼────────────────────┼─────────────────────────┼──────────────────────┼───────────────────┤
│ 2026-07-27 │ Ana Paula Santos         │ 123.456.789-01

In [17]:

query_str="""
    select * from trusted.rh_historico_trusted limit 10
"""
query_my_duckdb(query_str)

┌────────────┬──────────────────────┬────────────────┬────────────────────────────────────────┬─────────────────┬───────────────┬───────────┬─────────────────────┬─────────────────────────┬──────────────────────┬───────────────────┐
│  ref_data  │         Nome         │      CPF       │                Unidade                 │ Data_Nascimento │ Data_Admissao │  Status   │ Status_Afastamento  │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │ Data_Desligamento │
│    date    │       varchar        │    varchar     │                varchar                 │      date       │     date      │  varchar  │       varchar       │          date           │         date         │       date        │
├────────────┼──────────────────────┼────────────────┼────────────────────────────────────────┼─────────────────┼───────────────┼───────────┼─────────────────────┼─────────────────────────┼──────────────────────┼───────────────────┤
│ 2026-07-13 │ Ana Paula Santos     │ 123.456.789-01 │ Loja Brasília

In [32]:

query_str="""
select 
    ref_data,count(1) 
from trusted.rh_historico_trusted 
group by all 
limit 10
"""
query_my_duckdb(query_str)

┌────────────┬──────────┐
│  ref_data  │ count(1) │
│    date    │  int64   │
├────────────┼──────────┤
│ 2026-07-13 │       50 │
│ 2026-07-20 │       55 │
│ 2026-07-27 │       60 │
└────────────┴──────────┘



In [49]:
# Teste de volume

query_str="""

with atual as (
    select *
    from trusted.rh_trusted
),
anterior as (
    select *
    from trusted.rh_historico_trusted
    where ref_data = (select max(ref_data)-7 from trusted.rh_trusted)
),
vol_atual as (
    select ref_data, count(*) as registros from atual group by ref_data
),
vol_anterior as (
    select ref_data, count(*) as registros from anterior group by ref_data
)
select
    'volume_estagnado_ou_caiu' as alerta,
    va.ref_data as ref_data_atual,
    vo.ref_data as ref_data_anterior,
    va.registros as registros_atual,
    vo.registros as registros_anterior
from vol_atual va
cross join vol_anterior vo
where va.registros <= vo.registros
"""
query_my_duckdb(query_str)

┌─────────┬────────────────┬───────────────────┬─────────────────┬────────────────────┐
│ alerta  │ ref_data_atual │ ref_data_anterior │ registros_atual │ registros_anterior │
│ varchar │      date      │       date        │      int64      │       int64        │
└─────────┴────────────────┴───────────────────┴─────────────────┴────────────────────┘
                                        0 rows                                       



In [ ]:
# Teste de consistencia - Existian na anterior e sumiram na atual

query_str="""

with atual as (
    select *
    from trusted.rh_trusted
),
anterior as (
    select *
    from trusted.rh_historico_trusted
    where ref_data = (select max(ref_data)-7 from trusted.rh_trusted)
),
consolidado as (
    select * from atual
        union all
    select * from anterior
)


select * from (
    select 
        *
    from consolidado
    -- where ref_data 
    qualify count(distinct ref_data) over(partition by cpf) == 1
    order by nome
)
where ref_data = (select max(ref_data)-7 from trusted.rh_trusted)

"""
query_my_duckdb(query_str)

┌────────────┬──────────────────────┬────────────────┬──────────────────────────────┬─────────────────┬───────────────┬─────────┬────────────────────┬─────────────────────────┬──────────────────────┬───────────────────┐
│  ref_data  │         Nome         │      CPF       │           Unidade            │ Data_Nascimento │ Data_Admissao │ Status  │ Status_Afastamento │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │ Data_Desligamento │
│    date    │       varchar        │    varchar     │           varchar            │      date       │     date      │ varchar │      varchar       │          date           │         date         │       date        │
├────────────┼──────────────────────┼────────────────┼──────────────────────────────┼─────────────────┼───────────────┼─────────┼────────────────────┼─────────────────────────┼──────────────────────┼───────────────────┤
│ 2026-07-20 │ Carlos Eduardo Silva │ 987.654.321-02 │ Loja Manaus Atacarejo        │ 1990-07-22      │ 2026-03-12    │ 

In [48]:
# Teste de de intervalo afastamento

query_str="""

with atual as (
    select *
    from trusted.rh_trusted
)

select * from atual
where  Data_Inicio_Afastamento >= Data_Fim_Afastamento

"""
query_my_duckdb(query_str)

┌────────────┬───────────────┬────────────────┬──────────────────────────────────┬─────────────────┬───────────────┬─────────┬────────────────────┬─────────────────────────┬──────────────────────┬───────────────────┐
│  ref_data  │     Nome      │      CPF       │             Unidade              │ Data_Nascimento │ Data_Admissao │ Status  │ Status_Afastamento │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │ Data_Desligamento │
│    date    │    varchar    │    varchar     │             varchar              │      date       │     date      │ varchar │      varchar       │          date           │         date         │       date        │
├────────────┼───────────────┼────────────────┼──────────────────────────────────┼─────────────────┼───────────────┼─────────┼────────────────────┼─────────────────────────┼──────────────────────┼───────────────────┤
│ 2026-07-27 │ Thiago Nunes  │ 963.741.852-12 │ Loja Londrina Atacarejo          │ 1989-10-14      │ 2025-08-03    │ ATIVO   │ Féria

In [66]:
# Teste de de intervalo afastamento

query_str="""

with atual as (
    select *
        from trusted.rh_trusted
)

select * from atual
where  status = 'DESLIGADO' and Data_Desligamento is null

"""
query_my_duckdb(query_str)

┌────────────┬─────────────────┬────────────────┬───────────────────────────┬─────────────────┬───────────────┬───────────┬────────────────────┬─────────────────────────┬──────────────────────┬───────────────────┐
│  ref_data  │      Nome       │      CPF       │          Unidade          │ Data_Nascimento │ Data_Admissao │  Status   │ Status_Afastamento │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │ Data_Desligamento │
│    date    │     varchar     │    varchar     │          varchar          │      date       │     date      │  varchar  │      varchar       │          date           │         date         │       date        │
├────────────┼─────────────────┼────────────────┼───────────────────────────┼─────────────────┼───────────────┼───────────┼────────────────────┼─────────────────────────┼──────────────────────┼───────────────────┤
│ 2026-07-27 │ Camila Mendes   │ 852.963.741-11 │ Loja Brasília Express     │ 1990-05-25      │ 2026-04-12    │ DESLIGADO │ Desligado          │

In [73]:
# Teste de de intervalo afastamento

query_str="""

select * from "dbt_test__audit"."rh_desligados"

"""
query_my_duckdb(query_str)

┌────────────┬────────────────┬─────────────────┬───────────┬───────────────────┐
│  ref_data  │      CPF       │      Nome       │  Status   │ Data_Desligamento │
│    date    │    varchar     │     varchar     │  varchar  │       date        │
├────────────┼────────────────┼─────────────────┼───────────┼───────────────────┤
│ 2026-07-27 │ 852.963.741-11 │ Camila Mendes   │ DESLIGADO │ NULL              │
│ 2026-07-27 │ 654.321.987-19 │ Vanessa Pereira │ DESLIGADO │ NULL              │
└────────────┴────────────────┴─────────────────┴───────────┴───────────────────┘



In [78]:
# Teste de de intervalo afastamento

query_str="""

-- select * from "my-duckdb"."dbt_test__audit"."rh_desligados"
-- select * from "my-duckdb"."dbt_test__audit"."rh_sem_registro"
select * from "my-duckdb"."dbt_test__audit"."rh_intervalo_status"

"""
query_my_duckdb(query_str)

┌────────────┬────────────────┬───────────────┬────────────────────┬─────────────────────────┬──────────────────────┐
│  ref_data  │      CPF       │     Nome      │ Status_Afastamento │ Data_Inicio_Afastamento │ Data_Fim_Afastamento │
│    date    │    varchar     │    varchar    │      varchar       │          date           │         date         │
├────────────┼────────────────┼───────────────┼────────────────────┼─────────────────────────┼──────────────────────┤
│ 2026-07-27 │ 963.741.852-12 │ Thiago Nunes  │ Férias             │ 2026-03-05              │ 2026-02-05           │
│ 2026-07-27 │ 357.951.852-45 │ Gisele Santos │ Férias             │ 2026-03-20              │ 2026-03-15           │
│ 2026-07-27 │ 852.963.321-50 │ Renato Costa  │ Férias             │ 2026-03-10              │ 2026-02-10           │
└────────────┴────────────────┴───────────────┴────────────────────┴─────────────────────────┴──────────────────────┘

